In [21]:
from sklearn.model_selection import train_test_split

import pandas as pd
ddri = pd.read_csv('../lsy/ddri_prediction_long_train_2023_2024.csv')
ddri

# target 파악
ddri.rental_count.unique()

# feature 선정
ddri.columns
columns = ['hour', 'mapped_dong_code', 'cluster', 'weekday', 
           'month', 'holiday', 'temperature', 'humidity', 
           'precipitation', 'wind_speed']

# 담당한 군집 df 생성
ddri_morning = ddri[ddri.station_group == '아침 도착 업무 집중형']
ddri_morning[columns]

# feature, target
feature = ddri_morning[columns]
target = ddri_morning.rental_count

from sklearn.model_selection import train_test_split
train_data, test_data, train_target, test_target = train_test_split(
    feature, target, random_state=42
)

x_train, x_valid, y_train, y_valid = train_test_split(
    train_data, train_target, random_state=42
)

# StackingRegressor
# 1차 모델 - RandomForestRegressor, LGBMRegressor, XGBRegressor
# 2차 모델 - LinearRegressor

from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import StackingRegressor

# 1차 모델
estimators = [
    ('rf', RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1)),
    ('lgb', LGBMRegressor(
    verbose=-1, # 로그 출력 안함
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42)),
    ('xgb', XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42))
]

# 2차 모델
final_estimator = LinearRegression()

stack_model = StackingRegressor(
    estimators=estimators,
    final_estimator=final_estimator,
    cv=5,
    n_jobs=-1
)

stack_model.fit(train_data, train_target)

print(stack_model.score(train_data, train_target), stack_model.score(test_data, test_target))
print('-' * 40)

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

test_pred = stack_model.predict(test_data)

rmse = np.sqrt(mean_squared_error(test_target, test_pred))
mae = mean_absolute_error(test_target, test_pred)
r2 = r2_score(test_target, test_pred)

print('rmse:', rmse, '| mae:', mae, '| r2:', r2)



0.7764921597188579 0.6666996038223303
----------------------------------------
rmse: 1.5025987819856756 | mae: 0.8978568227115734 | r2: 0.6666996038223303


>상관계수 보기   
cluster는 feature로 쓸 수 있다고 일단 판단함

In [17]:
feature.corr()

,hour,cluster,mapped_dong_code,weekday,month,holiday,temperature,humidity,precipitation,wind_speed
hour,1.000000,NaN,0.000000,-0.000000,0.000000,-0.000000,0.121754,-0.257373,0.007863,0.163248
cluster,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mapped_dong_code,0.000000,NaN,1.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
weekday,-0.000000,NaN,-0.000000,1.000000,0.009509,-0.047665,0.001988,0.022001,-0.005666,0.002711
month,0.000000,NaN,0.000000,0.009509,1.000000,-0.043080,0.226226,0.196770,0.034101,-0.098899
holiday,-0.000000,NaN,0.000000,-0.047665,-0.043080,1.000000,-0.040807,0.000636,0.015949,0.015714
temperature,0.121754,NaN,0.000000,0.001988,0.226226,-0.040807,1.000000,0.158848,0.134162,0.016230
humidity,-0.257373,NaN,0.000000,0.022001,0.196770,0.000636,0.158848,1.000000,0.172910,-0.253358
precipitation,0.007863,NaN,0.000000,-0.005666,0.034101,0.015949,0.134162,0.172910,1.000000,0.154105
wind_speed,0.163248,NaN,0.000000,0.002711,-0.098899,0.015714,0.016230,-0.253358,0.154105,1.000000


> feature 간 상관이 약하다. feature를 더 찾아보면 좋을 것 같음   
근데 target 바뀐다고 함

In [23]:
ddri

,station_id,station_name,station_group,date,hour,rental_count,cluster,mapped_dong_code,weekday,month,holiday,temperature,humidity,precipitation,wind_speed
0,2312,청담역 13번 출구 앞,주거 도착형,2023-01-01,0,0,2,11680565,6,1,1,-2.000000,81,0.000000,9.300000
1,2312,청담역 13번 출구 앞,주거 도착형,2023-01-01,1,0,2,11680565,6,1,1,-1.700000,82,0.000000,9.300000
2,2312,청담역 13번 출구 앞,주거 도착형,2023-01-01,2,0,2,11680565,6,1,1,-0.800000,77,0.000000,10.800000
3,2312,청담역 13번 출구 앞,주거 도착형,2023-01-01,3,0,2,11680565,6,1,1,-1.800000,83,0.000000,8.000000
4,2312,청담역 13번 출구 앞,주거 도착형,2023-01-01,4,0,2,11680565,6,1,1,-4.000000,90,0.000000,8.600000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263155,4917,일원에코파크 주차장,주거 도착형,2024-12-31,19,1,2,11680730,1,12,0,-2.000000,65,0.000000,6.400000
263156,4917,일원에코파크 주차장,주거 도착형,2024-12-31,20,1,2,11680730,1,12,0,-2.500000,69,0.000000,4.900000
263157,4917,일원에코파크 주차장,주거 도착형,2024-12-31,21,1,2,11680730,1,12,0,-2.900000,70,0.000000,4.500000
263158,4917,일원에코파크 주차장,주거 도착형,2024-12-31,22,0,2,11680730,1,12,0,-2.900000,70,0.000000,3.200000
